# D100 — Iterator Basics

This notebook introduces Python iterables and iterators through small e-commerce examples.

## Learning goals

- Distinguish an **iterable** from an **iterator**.
- Understand what `iter()`, `next()`, and `StopIteration` do.
- Know what “back”, “last”, and exhaustion mean for an iterator.
- Use `range`, list, tuple, set, and dictionary iteration.
- See how a `for` loop consumes an iterator behind the scenes.
- Build a small custom iterator and a generator that each produce at most five values.
- Understand why iterators are useful for large or streaming data.

## 1. High-level idea

An **iterable** is an object we can loop over. It can provide a fresh iterator by calling `iter(iterable)`. Examples include a list, tuple, set, dictionary, string, and `range`.

An **iterator** is the object that remembers the current position and returns one item at a time. It supports:

- `iter(iterator)` — normally returns that same iterator.
- `next(iterator)` — returns the next item.
- `StopIteration` — signals that no items remain.

A Python `for` loop roughly performs these operations:

```python
iterator = iter(iterable)
while True:
    try:
        item = next(iterator)
        # execute the loop body
    except StopIteration:
        break
```

The loop catches `StopIteration` automatically, so we normally do not see it.

In [ ]:
# A list is iterable, but it is not itself an iterator.
products = ["laptop", "mouse", "keyboard"]
product_iterator = iter(products)

print(type(products))
print(type(product_iterator))
print("Does the list have __iter__?", hasattr(products, "__iter__"))
print("Does the list have __next__?", hasattr(products, "__next__"))
print("Does its iterator have __next__?", hasattr(product_iterator, "__next__"))

## 2. Basic collection examples

Each loop asks the collection for an iterator and then consumes its values. A list and tuple preserve their sequence. A set contains unique values, but its iteration order should not be relied upon. A dictionary iterates over keys by default; use `.items()` for key-value pairs.

In [ ]:
# range generates numbers as they are requested; it does not store a large list of them.
print("Order numbers from range:")
for order_number in range(1001, 1004):
    print(order_number)

print("\nProducts from a list:")
for product in ["phone", "charger", "case"]:
    print(product)

print("\nDelivery stages from a tuple:")
for stage in ("packed", "shipped", "delivered"):
    print(stage)

print("\nUnique categories from a set (order may vary):")
for category in {"books", "fashion", "grocery"}:
    print(category)

print("\nCart quantities from a dictionary:")
cart = {"book": 2, "pen": 5}
for product, quantity in cart.items():
    print(product, "->", quantity)

## 3. Manual use of `iter()` and `next()`

`next()` advances the iterator. The iterator is stateful: after returning an item, it remembers that the item has already been consumed.

In [ ]:
order_statuses = ["placed", "packed", "shipped"]
status_iterator = iter(order_statuses)

print(next(status_iterator))  # first item
print(next(status_iterator))  # second item
print(next(status_iterator))  # third item

try:
    print(next(status_iterator))  # there is no fourth item
except StopIteration:
    print("StopIteration: the iterator is exhausted")

In [ ]:
# A default value can avoid an exception when manually requesting an item.
coupon_iterator = iter(["SAVE10"])
print(next(coupon_iterator, "NO_MORE_COUPONS"))
print(next(coupon_iterator, "NO_MORE_COUPONS"))

### What about next, back, last, and stop?

- **Next:** `next(it)` moves forward by one item.
- **Back/previous:** Python's iterator protocol has no standard `back()` or `previous()`. Iterators are normally one-directional. If backward access is required, keep the data in a sequence such as a list and use indexes, or create a new iterator with `reversed(sequence)`.
- **Last:** there is no standard `last()` method. Consuming an iterator to find its last value also exhausts it. A sequence already in memory can use `items[-1]`.
- **Stop:** after the final item, `next(it)` raises `StopIteration`. A `for` loop catches it and stops normally. `break` lets the consumer stop early.
- **Restart:** an exhausted iterator does not reset itself. Call `iter()` on the original iterable to create a fresh iterator. Calling `iter()` on the exhausted iterator normally returns the same exhausted object.

In [ ]:
orders = ["ORD-1", "ORD-2", "ORD-3"]
orders_iterator = iter(orders)

print("Consume one:", next(orders_iterator))
print("Consume the rest:", list(orders_iterator))
print("Already exhausted:", list(orders_iterator))
print("Fresh iterator from original list:", list(iter(orders)))
print("Backward view of the list:", list(reversed(orders)))
print("Last item in the stored list:", orders[-1])

## 4. Consumer side and iterator side

There are two roles:

1. The **consumer** asks for values. A `for` loop, `next()`, `list()`, `sum()`, and many other tools can be consumers.
2. The **iterator/producer** remembers its state, prepares one value, returns it, and eventually raises `StopIteration`.

The consumer controls *when* another value is requested. The producer controls *how* that value is obtained. This separation allows lazy processing.

## 5. A custom iterator class (similar to a small `range`)

`OrderValueIterator` produces at most five calculated order values. It does not build a list first. Each call to `next()` calculates one value and prints a fetch message so we can see when the consumer requests data.

A custom iterator class implements:

- `__iter__()` to return the iterator object.
- `__next__()` to return one value or raise `StopIteration`.

In [ ]:
class OrderValueIterator:
    """Produce a limited sequence of simple calculated order values."""

    def __init__(self, start=100, step=25, count=5):
        if count < 0 or count > 5:
            raise ValueError("count must be between 0 and 5")
        self.start = start
        self.step = step
        self.count = count
        self.position = 0

    def __iter__(self):
        print("ITERATOR: __iter__() was called")
        return self

    def __next__(self):
        if self.position >= self.count:
            print("ITERATOR: no values remain; raising StopIteration")
            raise StopIteration

        value = self.start + self.position * self.step
        self.position += 1
        print(f"ITERATOR: fetched value {value} (item {self.position}/{self.count})")
        return value

In [ ]:
# Consumer side: the for loop asks for one value at a time.
order_values = OrderValueIterator(start=100, step=25, count=5)

for order_value in order_values:
    print(f"CONSUMER: received order value ₹{order_value}")

Observe the order of the output: the iterator fetch message appears first, then the consumer receives that value. After five items, the producer raises `StopIteration`; the `for` loop catches it and finishes.

In [ ]:
# A consumer can stop early. Values after the break are never calculated or fetched.
limited_values = OrderValueIterator(start=200, step=50, count=5)

for order_value in limited_values:
    print(f"CONSUMER: checking ₹{order_value}")
    if order_value >= 300:
        print("CONSUMER: enough data; stop early")
        break

Because the loop stopped early, the same iterator still has unconsumed items. Continuing it resumes from its saved position; it does not start again.

In [ ]:
print("Remaining values:", list(limited_values))

## 6. The same idea with a generator and `yield`

A **generator function** contains `yield`. Calling it creates a generator iterator but does not execute its body immediately. Each `next()` request resumes execution, produces one value at `yield`, and pauses while preserving local state. When the function finishes, Python automatically raises `StopIteration`.

`yield` is not required for every iterator—we built one using a class above—but it is usually the simpler way to write a lazy producer.

In [ ]:
def generate_order_discounts(count=5):
    """Yield at most five deterministic discount percentages."""
    if count < 0 or count > 5:
        raise ValueError("count must be between 0 and 5")

    for position in range(count):
        discount = 5 + position * 5
        print(f"GENERATOR: fetching discount {discount}%")
        yield discount

    print("GENERATOR: finished; StopIteration happens automatically")

In [ ]:
discount_iterator = generate_order_discounts(5)
print("Created generator; its body has not run yet.\n")

for discount in discount_iterator:
    print(f"CONSUMER: applying {discount}% discount")

### Optional random values

The next generator yields five random sample order values. A fixed seed is used only to make the lesson repeatable. The values are still generated one at a time rather than stored in advance.

In [ ]:
import random

def random_order_values(count=5, seed=42):
    if count < 0 or count > 5:
        raise ValueError("count must be between 0 and 5")

    random_generator = random.Random(seed)
    for order_number in range(1, count + 1):
        value = random_generator.randint(100, 500)
        print(f"PRODUCER: fetched random value for order {order_number}")
        yield {"order_id": f"ORD-{order_number}", "value": value}

for order in random_order_values():
    print("CONSUMER:", order)

## 7. Why use iterators?

- **Lower memory use:** process one item at a time instead of storing every item in a list.
- **Lazy work:** calculate or fetch a value only when a consumer requests it.
- **Early stopping:** avoid producing unused values when the consumer uses `break`.
- **Streams and very large data:** useful for files, database results, API pages, events, and data pipelines.
- **Composable processing:** iterator tools can form pipelines without creating an intermediate list at every step.

The trade-off is that most iterators are forward-only and single-use. If repeated access, indexing, or backward movement is important—and the data fits in memory—a list may be more suitable.

In [ ]:
# A lazy pipeline: no result list is required until list(...) consumes it.
prices = [80, 120, 250, 40]
expensive_prices = filter(lambda price: price >= 100, prices)
discounted_prices = map(lambda price: price * 0.9, expensive_prices)

print("Is filter an iterator?", iter(expensive_prices) is expensive_prices)
print("Final values:", list(discounted_prices))

## 8. Common iterable and iterator objects

| Object | Iterable? | Iterator itself? | Notes |
|---|---:|---:|---|
| list, tuple, set, dictionary, string | Yes | No | Usually creates a fresh iterator for each loop |
| `range(...)` | Yes | No | Compact, lazy-style numeric sequence; reusable |
| `iter(collection)` | Yes | Yes | Stateful, normally single-use |
| generator expression / generator function result | Yes | Yes | Uses lazy evaluation |
| `enumerate`, `zip`, `map`, `filter` results | Yes | Yes | Common lazy iterator tools in Python 3 |
| open file object | Yes | Yes | Iterating reads successive lines; close it after use |

A quick practical test is `iter(obj) is obj`: this is normally `True` for an iterator and `False` for a reusable collection.

## 9. Summary

- An iterable can create an iterator; an iterator delivers one item at a time.
- `iter()` obtains the iterator and `next()` advances it.
- `StopIteration` means that the iterator is exhausted.
- A `for` loop performs these steps automatically.
- Standard iterators move forward; they do not provide general `back()` or `last()` operations.
- A custom class uses `__iter__()` and `__next__()`; a generator uses `yield`.
- Iterators make memory-efficient, lazy, streaming, and early-stop processing possible.

## 10. Quick practice

1. Create an iterator from `['new', 'paid', 'packed']` and call `next()` manually until it is exhausted.
2. Change `OrderValueIterator` to produce only three values.
3. Stop `random_order_values()` after receiving an order whose value is greater than ₹300.
4. Write a generator named `order_ids()` that yields `ORD-1` through `ORD-5`.